# Bailey Supplementary: Theta Connectivity & Alpha Power Replication Check

**Objective.** Test Bailey et al.'s (2019) theta-connectivity and alpha-power constructs for association with rTMS responder status in the TDBRAIN cohort, as an independent supplementary replication check - not part of the primary arm comparison (Decision 5, `modelling_decisions.md`).

**Context.** Bailey et al. (2019, J Affect Disord) reported that responders showed increased theta connectivity (wPLI) across a 66-pair frontal-parietal-occipital network, and reduced alpha power at F3/F4/O1/O2, in a small exploratory sample. An independent replication (Bailey, Krepel & van Dijk, 2021, Clin Neurophysiol, N=193) found no significant difference in either measure, using a reduced 14-pair subset of the original network (limited by montage overlap between the two datasets), with the theta connectivity effect in the *opposite* direction to the original finding.

TDBRAIN's montage contains all 8 electrodes required for the full 14-pair set (Fz, FC3, FC4, F3, P3, P4, O1, O2 - confirmed against the pilot subject's channel list), so this check is not limited to a further-reduced subset the way the 2021 replication was.

**Inputs.**
- 14 electrode pairs (theta wPLI), verified from 2021 paper source text: Fz-FC4, FC3-FC4, F3-P3, FC3-P3, P3-P4, F3-O1, FC3-O1, P3-O1, P4-O1, F3-O2, FC3-O2, P3-O2, P4-O2, O1-O2
- Alpha power at F3, F4, O1, O2 (2019 paper's electrode set, restated in 2021 paper's methods)
- PLI substitute for wPLI (Decision: primary-pool connectivity metric already justified on volume-conduction grounds; no coherence/PLV extension for this supplementary check - out of scope)
- Responder/non-responder labels, age (for fold-scoped deconfounding, consistent with all other arms)

**Assumptions.**
- This construct has one prior positive result (2019, exploratory, small N) and one prior full-strength negative result (2021, N=193, reversed-direction effect). TDBRAIN is a third independent sample; a null result here would add to, not merely repeat, existing negative evidence. A positive result would need to be weighed against two more heavily-powered/replication-focused studies rather than treated as a novel discovery.
- PLI is used in place of wPLI (weighted PLI) as the connectivity metric, consistent with this project's primary-pool choice - a departure from exact replication, documented here rather than treated as equivalent without comment.

In [1]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
from pathlib import Path
from scipy import stats

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, balanced_accuracy_score

from mne_connectivity import spectral_connectivity_epochs

# Temporary bootstrap path, just to make src/ importable - matches Arm 1's pattern
sys.path.insert(0, str(Path.cwd().parent))

from src.preprocessing import find_repo_root
from src.features import load_subject_epochs, get_subject_qc, compute_pli
from src.modelling import AgeDeconfounder, run_nested_cv, paired_comparison_test

project_root = find_repo_root()
data_dir = project_root / "data"
features_dir = data_dir / "features"

In [2]:
BAILEY_PAIRS = [
    ('Fz', 'FC4'), ('FC3', 'FC4'), ('F3', 'P3'), ('FC3', 'P3'), ('P3', 'P4'),
    ('F3', 'O1'), ('FC3', 'O1'), ('P3', 'O1'), ('P4', 'O1'),
    ('F3', 'O2'), ('FC3', 'O2'), ('P3', 'O2'), ('P4', 'O2'), ('O1', 'O2'),
]  # verified from source: Bailey, Krepel & van Dijk (2021), Clin Neurophysiol
   # (14 pairs = original 66-pair network reduced to electrodes present in
   # both the 2019 and 2021 datasets; TDBRAIN separately confirmed to contain
   # all 8 electrodes needed, so no further reduction happens here)

THETA_BAND = {'theta': (4, 8)}  # standard theta range, matching Bailey et al.

In [3]:
## Funcitons for computing Bailey's theta connectivity measure
def compute_bailey_theta_pli(epochs, pairs, band):
    """
    Compute PLI for a fixed set of named electrode pairs, in a single band,
    averaged into one value per subject - matching Bailey et al.'s (2019,
    2021) averaged theta wPLI summary measure (Table 1), not a per-pair
    feature set. PLI substitutes for wPLI here (Decision: project's primary
    connectivity metric, volume-conduction rationale) - not an exact
    replication of their weighting.

    Parameters:
    epochs (mne.Epochs): The preprocessed epochs
    pairs (list of (str, str)): Named electrode pairs, e.g. BAILEY_PAIRS
    band (dict): single band_name: (low, high) - must contain exactly one band

    Returns:
    result (dict): {'bailey_theta_pli_avg': mean PLI across all named pairs}
    """
    # This function takes one band, not a bands dict like compute_pli - Bailey's
    # measure is theta-only
    assert len(band) == 1, "compute_bailey_theta_pli expects exactly one band"
    ch_names = epochs.ch_names

    # Fail loudly if a named electrode isn't in this subject's montage
    missing = [ch for pair in pairs for ch in pair if ch not in ch_names]
    assert not missing, f"channels not found in epochs: {set(missing)}"

    # Build seed/target index lists the same way compute_pli does for its
    # all-pairs case, but from the fixed named list instead of combinations().
    seeds = [ch_names.index(a) for a, b in pairs]
    targets = [ch_names.index(b) for a, b in pairs]
    indices = (seeds, targets)

    # Unpack the single (name, (low, high)) tuple 
    (band_name, (low, high)), = band.items()

    con = spectral_connectivity_epochs(
        data=epochs,
        method='pli',
        indices=indices,
        fmin=low,
        fmax=high,
        faverage=True,
        sfreq=epochs.info['sfreq'],
        )
    con_data = con.get_data()  # shape (n_pairs, 1) - one value per named pair
    pair_values = con_data[:, 0]

    # Same validity checks as compute_pli: PLI is bounded [0,1] by definition,
    # and a NaN here usually means a channel had zero variance or bad data.
    assert not np.any(np.isnan(pair_values)), "NaN found in pair-level PLI values"
    assert np.all((pair_values >= 0) & (pair_values <= 1)), "PLI value outside [0,1] found"

    # Average across all 14 pairs into one subject-level feature - this is the
    # deliberate departure from compute_pli's per-pair output, matching Bailey's
    # single averaged connectivity measure rather than a per-pair feature bank.
    result = {f"bailey_{band_name}_pli_avg": pair_values.mean()}
    return result


def extract_bailey_subject(subject_id, data_dir, qc_log):
    """
    Extract Bailey theta-connectivity feature for one subject, restEC/heog_off
    (matching Arm 1's condition/variant).
    """
    try:
        # restEC/heog_off is hardcoded, not a parameter - this check is
        # explicitly scoped to Arm 1's condition, not a general-purpose
        # multi-condition extractor like extract_subject_features.
        epochs = load_subject_epochs(subject_id, 'restEC', 'heog_off', data_dir)
        qc_info = get_subject_qc(subject_id, 'restEC', 'heog_off', qc_log)
        theta_pli = compute_bailey_theta_pli(epochs, BAILEY_PAIRS, THETA_BAND)
        results = {
            "subject_id": subject_id,
            "reason": "ok",
            **theta_pli,
        }
    except Exception as e:
        # Same fail-soft pattern as extract_subject_features: one bad subject
        # doesn't kill the whole cohort loop, and the failure reason is kept
        # for QC review afterward rather than swallowed.
        results = {
            "subject_id": subject_id,
            "reason": str(e),
        }
    return results

In [4]:
# Validate compute_bailey_theta_pli against the already-validated compute_pli,
# using one pilot subject. compute_pli generates all-pairs indices via
# combinations(); compute_bailey_theta_pli looks up 14 named pairs directly -
# different code paths that should still agree exactly on those 14 pairs.

pilot_subject = 'sub-87999321'  # matches the pilot subject used in 02_preprocessing_pilot.ipynb
pilot_epochs = load_subject_epochs(pilot_subject, 'restEC', 'heog_off', data_dir)

# Ground truth: compute theta PLI for every channel pair via the existing function
all_pairs_pli = compute_pli(pilot_epochs, THETA_BAND)

# Pull out just the 14 Bailey pairs from that all-pairs output.
# compute_pli's keys are f"{ch_a}_{ch_b}_{band}_pli", built from combinations()
# in electrode-index order - so a pair may appear as either A_B or B_A depending
# on the two channels' relative positions in epochs.ch_names. Check both orders.
manual_values = []
for a, b in BAILEY_PAIRS:
    key_ab = f"{a}_{b}_theta_pli"
    key_ba = f"{b}_{a}_theta_pli"
    if key_ab in all_pairs_pli:
        manual_values.append(all_pairs_pli[key_ab])
    elif key_ba in all_pairs_pli:
        manual_values.append(all_pairs_pli[key_ba])
    else:
        raise KeyError(f"Neither {key_ab} nor {key_ba} found in compute_pli output")

manual_avg = np.mean(manual_values)

# New function's output for the same subject
new_function_result = compute_bailey_theta_pli(pilot_epochs, BAILEY_PAIRS, THETA_BAND)
new_avg = new_function_result['bailey_theta_pli_avg']

print(f"Ground truth (from compute_pli, averaged over 14 named pairs): {manual_avg:.6f}")
print(f"compute_bailey_theta_pli output:                               {new_avg:.6f}")
print(f"Match: {np.isclose(manual_avg, new_avg)}")

assert np.isclose(manual_avg, new_avg), "Mismatch between compute_pli-derived average and compute_bailey_theta_pli - do not proceed to full cohort until resolved"

Reading /Users/romyweinstock/eeg-rtms-response-prediction/data/derivatives_heog_off/sub-87999321/sub-87999321_restEC-epo.fif ...
    Found the data of interest:
        t =       0.00 ...    4998.00 ms
        0 CTF compensation matrices available
Not setting metadata
23 matching events found
No baseline correction applied
0 projection items activated
Connectivity computation...
    using t=0.000s..4.998s for estimation (2500 points)
    frequencies: 4.0Hz..8.0Hz (21 points)
    connectivity scores will be averaged for each band
    computing connectivity for 325 connections
    Using multitaper spectrum estimation with 7 DPSS windows
    the following metrics will be computed: PLI
    computing cross-spectral density for epoch 1
    computing cross-spectral density for epoch 2
    computing cross-spectral density for epoch 3
    computing cross-spectral density for epoch 4
    computing cross-spectral density for epoch 5
    computing cross-spectral density for epoch 6
    computing c

/Users/romyweinstock/eeg-rtms-response-prediction/src/features.py:109: RuntimeWarning: There were no Annotations stored in <EpochsFIF | 23 events (all good), 0 – 4.998 s (baseline off), ~13.6 MiB, data loaded,
 '1': 23>, so metadata was not modified.
  con = spectral_connectivity_epochs(
/var/folders/sj/n3g071kn3k5d875m963jywkr0000gn/T/ipykernel_79021/2228109301.py:37: RuntimeWarning: There were no Annotations stored in <EpochsFIF | 23 events (all good), 0 – 4.998 s (baseline off), ~13.6 MiB, data loaded,
 '1': 23>, so metadata was not modified.
  con = spectral_connectivity_epochs(


In [5]:
# Full-cohort extraction: Bailey theta PLI, restEC/heog_off, matching Arm 1's cohort

qc_log = pd.read_csv(data_dir / "batch_results_log_full_cohort.csv")

bailey_subjects = qc_log[
    (qc_log['condition'] == 'restEC') &
    (qc_log['heog_variant'] == 'heog_off') &
    (qc_log['status'] == 'ok')
]['subject_id'].tolist()

print(len(bailey_subjects))  # expect 160, matching Arm 1/5/6's cohort size

bailey_rows = [extract_bailey_subject(sid, data_dir, qc_log) for sid in bailey_subjects]
bailey_df = pd.DataFrame(bailey_rows)

print(bailey_df['reason'].value_counts())  # expect all 'ok', or a small number of named failures
bailey_df.head()

160
Reading /Users/romyweinstock/eeg-rtms-response-prediction/data/derivatives_heog_off/sub-87999321/sub-87999321_restEC-epo.fif ...
    Found the data of interest:
        t =       0.00 ...    4998.00 ms
        0 CTF compensation matrices available
Not setting metadata
23 matching events found
No baseline correction applied
0 projection items activated
Connectivity computation...
    using t=0.000s..4.998s for estimation (2500 points)
    frequencies: 4.0Hz..8.0Hz (21 points)
    connectivity scores will be averaged for each band
    computing connectivity for 14 connections
    Using multitaper spectrum estimation with 7 DPSS windows
    the following metrics will be computed: PLI
    computing cross-spectral density for epoch 1
    computing cross-spectral density for epoch 2
    computing cross-spectral density for epoch 3
    computing cross-spectral density for epoch 4
    computing cross-spectral density for epoch 5
    computing cross-spectral density for epoch 6
    computin

/var/folders/sj/n3g071kn3k5d875m963jywkr0000gn/T/ipykernel_79021/2228109301.py:37: RuntimeWarning: There were no Annotations stored in <EpochsFIF | 23 events (all good), 0 – 4.998 s (baseline off), ~13.6 MiB, data loaded,
 '1': 23>, so metadata was not modified.
  con = spectral_connectivity_epochs(
/var/folders/sj/n3g071kn3k5d875m963jywkr0000gn/T/ipykernel_79021/2228109301.py:37: RuntimeWarning: There were no Annotations stored in <EpochsFIF | 23 events (all good), 0 – 4.998 s (baseline off), ~13.6 MiB, data loaded,
 '1': 23>, so metadata was not modified.
  con = spectral_connectivity_epochs(
/var/folders/sj/n3g071kn3k5d875m963jywkr0000gn/T/ipykernel_79021/2228109301.py:37: RuntimeWarning: There were no Annotations stored in <EpochsFIF | 24 events (all good), 0 – 4.998 s (baseline off), ~14.2 MiB, data loaded,
 '1': 24>, so metadata was not modified.
  con = spectral_connectivity_epochs(


    computing cross-spectral density for epoch 7
    computing cross-spectral density for epoch 8
    computing cross-spectral density for epoch 9
    computing cross-spectral density for epoch 10
    computing cross-spectral density for epoch 11
    computing cross-spectral density for epoch 12
    computing cross-spectral density for epoch 13
    computing cross-spectral density for epoch 14
    computing cross-spectral density for epoch 15
    computing cross-spectral density for epoch 16
    computing cross-spectral density for epoch 17
    computing cross-spectral density for epoch 18
    computing cross-spectral density for epoch 19
    computing cross-spectral density for epoch 20
    computing cross-spectral density for epoch 21
    computing cross-spectral density for epoch 22
    computing cross-spectral density for epoch 23
    computing cross-spectral density for epoch 24
[Connectivity computation done]
Reading /Users/romyweinstock/eeg-rtms-response-prediction/data/derivati

/var/folders/sj/n3g071kn3k5d875m963jywkr0000gn/T/ipykernel_79021/2228109301.py:37: RuntimeWarning: There were no Annotations stored in <EpochsFIF | 24 events (all good), 0 – 4.998 s (baseline off), ~14.2 MiB, data loaded,
 '1': 24>, so metadata was not modified.
  con = spectral_connectivity_epochs(
/var/folders/sj/n3g071kn3k5d875m963jywkr0000gn/T/ipykernel_79021/2228109301.py:37: RuntimeWarning: There were no Annotations stored in <EpochsFIF | 23 events (all good), 0 – 4.998 s (baseline off), ~13.6 MiB, data loaded,
 '1': 23>, so metadata was not modified.
  con = spectral_connectivity_epochs(
/var/folders/sj/n3g071kn3k5d875m963jywkr0000gn/T/ipykernel_79021/2228109301.py:37: RuntimeWarning: There were no Annotations stored in <EpochsFIF | 24 events (all good), 0 – 4.998 s (baseline off), ~14.2 MiB, data loaded,
 '1': 24>, so metadata was not modified.
  con = spectral_connectivity_epochs(
/var/folders/sj/n3g071kn3k5d875m963jywkr0000gn/T/ipykernel_79021/2228109301.py:37: RuntimeWarning

    computing cross-spectral density for epoch 2
    computing cross-spectral density for epoch 3
    computing cross-spectral density for epoch 4
    computing cross-spectral density for epoch 5
    computing cross-spectral density for epoch 6
    computing cross-spectral density for epoch 7
    computing cross-spectral density for epoch 8
    computing cross-spectral density for epoch 9
    computing cross-spectral density for epoch 10
    computing cross-spectral density for epoch 11
    computing cross-spectral density for epoch 12
    computing cross-spectral density for epoch 13
    computing cross-spectral density for epoch 14
    computing cross-spectral density for epoch 15
    computing cross-spectral density for epoch 16
    computing cross-spectral density for epoch 17
    computing cross-spectral density for epoch 18
    computing cross-spectral density for epoch 19
    computing cross-spectral density for epoch 20
    computing cross-spectral density for epoch 21
    comp

...

    computing cross-spectral density for epoch 15
    computing cross-spectral density for epoch 16
    computing cross-spectral density for epoch 17
    computing cross-spectral density for epoch 18
    computing cross-spectral density for epoch 19
    computing cross-spectral density for epoch 20
    computing cross-spectral density for epoch 21
    computing cross-spectral density for epoch 22
[Connectivity computation done]
Reading /Users/romyweinstock/eeg-rtms-response-prediction/data/derivatives_heog_off/sub-88077569/sub-88077569_restEC-epo.fif ...
    Found the data of interest:
        t =       0.00 ...    4998.00 ms
        0 CTF compensation matrices available
Not setting metadata
24 matching events found
No baseline correction applied
0 projection items activated
Connectivity computation...
    using t=0.000s..4.998s for estimation (2500 points)
    frequencies: 4.0Hz..8.0Hz (21 points)
    connectivity scores will be averaged for each band
    computing connectivity for 14

/var/folders/sj/n3g071kn3k5d875m963jywkr0000gn/T/ipykernel_79021/2228109301.py:37: RuntimeWarning: There were no Annotations stored in <EpochsFIF | 24 events (all good), 0 – 4.998 s (baseline off), ~14.2 MiB, data loaded,
 '1': 24>, so metadata was not modified.
  con = spectral_connectivity_epochs(
/var/folders/sj/n3g071kn3k5d875m963jywkr0000gn/T/ipykernel_79021/2228109301.py:37: RuntimeWarning: There were no Annotations stored in <EpochsFIF | 23 events (all good), 0 – 4.998 s (baseline off), ~13.6 MiB, data loaded,
 '1': 23>, so metadata was not modified.
  con = spectral_connectivity_epochs(
/var/folders/sj/n3g071kn3k5d875m963jywkr0000gn/T/ipykernel_79021/2228109301.py:37: RuntimeWarning: There were no Annotations stored in <EpochsFIF | 23 events (all good), 0 – 4.998 s (baseline off), ~13.6 MiB, data loaded,
 '1': 23>, so metadata was not modified.
  con = spectral_connectivity_epochs(


[Connectivity computation done]
Reading /Users/romyweinstock/eeg-rtms-response-prediction/data/derivatives_heog_off/sub-88026005/sub-88026005_restEC-epo.fif ...
    Found the data of interest:
        t =       0.00 ...    4998.00 ms
        0 CTF compensation matrices available
Not setting metadata
23 matching events found
No baseline correction applied
0 projection items activated
Connectivity computation...
    using t=0.000s..4.998s for estimation (2500 points)
    frequencies: 4.0Hz..8.0Hz (21 points)
    connectivity scores will be averaged for each band
    computing connectivity for 14 connections
    Using multitaper spectrum estimation with 7 DPSS windows
    the following metrics will be computed: PLI
    computing cross-spectral density for epoch 1
    computing cross-spectral density for epoch 2
    computing cross-spectral density for epoch 3
    computing cross-spectral density for epoch 4
    computing cross-spectral density for epoch 5
    computing cross-spectral dens

/var/folders/sj/n3g071kn3k5d875m963jywkr0000gn/T/ipykernel_79021/2228109301.py:37: RuntimeWarning: There were no Annotations stored in <EpochsFIF | 23 events (all good), 0 – 4.998 s (baseline off), ~13.6 MiB, data loaded,
 '1': 23>, so metadata was not modified.
  con = spectral_connectivity_epochs(
/var/folders/sj/n3g071kn3k5d875m963jywkr0000gn/T/ipykernel_79021/2228109301.py:37: RuntimeWarning: There were no Annotations stored in <EpochsFIF | 23 events (all good), 0 – 4.998 s (baseline off), ~13.6 MiB, data loaded,
 '1': 23>, so metadata was not modified.
  con = spectral_connectivity_epochs(


,subject_id,reason,bailey_theta_pli_avg
0,sub-87999321,ok,0.320911
1,sub-88049537,ok,0.188997
2,sub-88049857,ok,0.258787
3,sub-88049905,ok,0.159014
4,sub-88050713,ok,0.225673


In [6]:
# Build bailey_model_df: theta PLI + age + Responder, same merge pattern as Arm 5/6

cohort_df = pd.read_excel(data_dir / "cohort_filtered_n163.xlsx")

bailey_model_df = bailey_df.merge(
    cohort_df[['TDBRAIN_ID', 'age', 'Responder']],
    left_on='subject_id', right_on='TDBRAIN_ID', how='left'
).drop(columns='TDBRAIN_ID')

print(len(bailey_model_df))                                              # expect 160
print(bailey_model_df[['bailey_theta_pli_avg', 'age', 'Responder']].isna().sum())  # expect all 0
print(bailey_model_df['Responder'].value_counts())                       # expect ~93/67, matching Arm 5/6
bailey_model_df.head()

160
bailey_theta_pli_avg    0
age                     0
Responder               0
dtype: int64
Responder
1    93
0    67
Name: count, dtype: int64


,subject_id,reason,bailey_theta_pli_avg,age,Responder
0,sub-87999321,ok,0.320911,49.66,1
1,sub-88049537,ok,0.188997,43.07,1
2,sub-88049857,ok,0.258787,44.62,1
3,sub-88049905,ok,0.159014,46.02,1
4,sub-88050713,ok,0.225673,32.77,1


In [7]:
#Run nested CV on Bailey's theta PLI, restEC/heog_off, with age deconfounding and balanced classifiers

N_OUTER_SPLITS = 5
N_INNER_SPLITS = 3
RANDOM_STATE = 42

X_bailey = bailey_model_df[['bailey_theta_pli_avg']].values
y_bailey = bailey_model_df['Responder'].values
age_bailey = bailey_model_df['age'].values

classifier_specs_balanced = {
    'LDA (Ledoit-Wolf, balanced priors)': (
        LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto', priors=[0.5, 0.5]),
        None
    ),
    'Logistic (unregularized, balanced)': (
        LogisticRegression(C=np.inf, max_iter=1000, class_weight='balanced'),
        None
    ),
    'Logistic (elastic-net, balanced)': (
        LogisticRegression(solver='saga', max_iter=5000, class_weight='balanced', random_state=RANDOM_STATE),
        {'C': [0.01, 0.1, 1, 10, 100], 'l1_ratio': [0.1, 0.5, 0.9]}
    ),
    'Logistic (L2 / "Bayesian" MAP, balanced)': (
        LogisticRegression(l1_ratio=0, max_iter=1000, class_weight='balanced'),
        {'C': [0.01, 0.1, 1, 10, 100]}
    ),
}

bailey_results_df = run_nested_cv(X_bailey, y_bailey, age_bailey, classifier_specs_balanced,
                                   N_OUTER_SPLITS, N_INNER_SPLITS, RANDOM_STATE)

print(bailey_results_df.groupby('classifier')[['balanced_accuracy', 'accuracy', 'auc', 'sensitivity', 'specificity', 'ppv']].mean())
bailey_results_df

                                          balanced_accuracy  accuracy  \
classifier                                                              
LDA (Ledoit-Wolf, balanced priors)                 0.425252    0.4125   
Logistic (L2 / "Bayesian" MAP, balanced)           0.425252    0.4125   
Logistic (elastic-net, balanced)                   0.432540    0.4875   
Logistic (unregularized, balanced)                 0.425252    0.4125   

                                               auc  sensitivity  specificity  \
classifier                                                                     
LDA (Ledoit-Wolf, balanced priors)        0.433433     0.353801     0.496703   
Logistic (L2 / "Bayesian" MAP, balanced)  0.433433     0.353801     0.496703   
Logistic (elastic-net, balanced)          0.448413     0.722222     0.142857   
Logistic (unregularized, balanced)        0.433433     0.353801     0.496703   

                                               ppv  
classifier                 

,classifier,fold,balanced_accuracy,accuracy,auc,sensitivity,specificity,ppv
0,"LDA (Ledoit-Wolf, balanced priors)",0,0.477733,0.43750,0.578947,0.263158,0.692308,0.555556
1,"Logistic (unregularized, balanced)",0,0.477733,0.43750,0.578947,0.263158,0.692308,0.555556
2,"Logistic (elastic-net, balanced)",0,0.500000,0.59375,0.500000,1.000000,0.000000,0.593750
3,"Logistic (L2 / ""Bayesian"" MAP, balanced)",0,0.477733,0.43750,0.578947,0.263158,0.692308,0.555556
4,"LDA (Ledoit-Wolf, balanced priors)",1,0.518219,0.50000,0.457490,0.421053,0.615385,0.615385
5,"Logistic (unregularized, balanced)",1,0.518219,0.50000,0.457490,0.421053,0.615385,0.615385
6,"Logistic (elastic-net, balanced)",1,0.500000,0.59375,0.500000,1.000000,0.000000,0.593750
7,"Logistic (L2 / ""Bayesian"" MAP, balanced)",1,0.518219,0.50000,0.457490,0.421053,0.615385,0.615385
8,"LDA (Ledoit-Wolf, balanced priors)",2,0.467611,0.46875,0.388664,0.473684,0.461538,0.562500
9,"Logistic (unregularized, balanced)",2,0.467611,0.46875,0.388664,0.473684,0.461538,0.562500


In [8]:
# Diagnostic: what regularization strength is elastic-net selecting per fold?
outer_cv = StratifiedKFold(n_splits=N_OUTER_SPLITS, shuffle=True, random_state=RANDOM_STATE)

for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X_bailey, y_bailey)):
    X_train, X_test = X_bailey[train_idx], X_bailey[test_idx]
    y_train, y_test = y_bailey[train_idx], y_bailey[test_idx]
    age_train, age_test = age_bailey[train_idx], age_bailey[test_idx]

    deconf = AgeDeconfounder()
    deconf.fit(X_train, age_train)
    X_train_clean = deconf.transform(X_train, age_train)

    estimator, param_grid = classifier_specs_balanced['Logistic (elastic-net, balanced)']
    inner_cv = StratifiedKFold(n_splits=N_INNER_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    search = GridSearchCV(estimator, param_grid, cv=inner_cv, scoring='balanced_accuracy')
    search.fit(X_train_clean, y_train)

    print(f"fold {fold_idx}: best_params={search.best_params_}, "
          f"y_train balance={np.bincount(y_train)}, "
          f"y_test balance={np.bincount(y_test)}")

fold 0: best_params={'C': 0.01, 'l1_ratio': 0.1}, y_train balance=[54 74], y_test balance=[13 19]
fold 1: best_params={'C': 0.01, 'l1_ratio': 0.1}, y_train balance=[54 74], y_test balance=[13 19]
fold 2: best_params={'C': 1, 'l1_ratio': 0.1}, y_train balance=[54 74], y_test balance=[13 19]
fold 3: best_params={'C': 100, 'l1_ratio': 0.1}, y_train balance=[53 75], y_test balance=[14 18]
fold 4: best_params={'C': 10, 'l1_ratio': 0.1}, y_train balance=[53 75], y_test balance=[14 18]


In [9]:
# Permutation test: Bailey supplementary (theta PLI)
N_PERMUTATIONS = 1000
rng = np.random.RandomState(RANDOM_STATE)

observed_bailey = bailey_results_df.groupby('classifier')['balanced_accuracy'].mean().to_dict()

print(f"N_PERMUTATIONS = {N_PERMUTATIONS}")

null_scores_bailey = {name: [] for name in classifier_specs_balanced}

for i in range(N_PERMUTATIONS):
    y_shuffled = rng.permutation(y_bailey)
    perm_df = run_nested_cv(X_bailey, y_shuffled, age_bailey, classifier_specs_balanced, N_OUTER_SPLITS, N_INNER_SPLITS, RANDOM_STATE)
    perm_result = perm_df.groupby('classifier')['balanced_accuracy'].mean().to_dict()
    for name, score in perm_result.items():
        null_scores_bailey[name].append(score)

print(f"{'classifier':<45} {'observed':>10} {'null mean':>10} {'p-value':>10}")
for name in classifier_specs_balanced:
    null_arr = np.array(null_scores_bailey[name])
    p_value = (np.sum(null_arr >= observed_bailey[name]) + 1) / (N_PERMUTATIONS + 1)
    print(f"{name:<45} {observed_bailey[name]:>10.4f} {null_arr.mean():>10.4f} {p_value:>10.4f}")

N_PERMUTATIONS = 1000
classifier                                      observed  null mean    p-value
LDA (Ledoit-Wolf, balanced priors)                0.4253     0.5028     0.9271
Logistic (unregularized, balanced)                0.4253     0.5028     0.9261
Logistic (elastic-net, balanced)                  0.4325     0.5027     0.9800
Logistic (L2 / "Bayesian" MAP, balanced)          0.4253     0.5026     0.9241


## Bailey Supplementary (theta PLI) - Findings

Permutation test (N=1000) against chance-level balanced accuracy, same nested-CV configuration as the observed run (5 outer / 3 inner folds, random_state=42):

| Classifier | Observed BA | Null mean | p-value |
|---|---|---|---|
| LDA (Ledoit-Wolf, balanced priors) | 0.4253 | 0.5028 | 0.9271 |
| Logistic (unregularized, balanced) | 0.4253 | 0.5028 | 0.9261 |
| Logistic (elastic-net, balanced) | 0.4325 | 0.5027 | 0.9800 |
| Logistic (L2, balanced) | 0.4253 | 0.5026 | 0.9241 |

No classifier exceeds chance; observed balanced accuracy sits below 0.5 for all four. Null means cluster tightly around 0.50, consistent with a correctly calibrated permutation procedure - including for elastic-net, despite the fold-level hyperparameter instability noted below.

Elastic-net's per-fold hyperparameter selection was unstable during the observed run (`C` ranging from 0.01 to 100 across the 5 outer folds), unlike the stable convergence seen in Arm 1/5/6. Diagnosed directly: in folds where `GridSearchCV`'s inner-CV balanced accuracy tied across the hyperparameter grid, sklearn's tie-breaking selected the first-evaluated (`C=0.01`) combination, producing a degenerate all-responder classifier (sensitivity=1, specificity=0) in those folds. This is a different phenomenon from the shared-grid-corner convergence diagnosed in Arm 1, and is itself consistent with `bailey_theta_pli_avg` carrying little to no signal in this cohort under this feature/classifier combination - not a bug in the pipeline.

This result is a third independent null: Bailey et al.'s theta connectivity construct showed a positive result only in the original small exploratory sample (2019), failed to replicate in a larger independent sample (2021, N=193, reversed-direction effect), and now also shows no association in TDBRAIN (N=160), a dataset whose montage supports the full 14-pair electrode set (unlike the 2021 replication's reduced coverage).

**Residual assumptions**

PLI is used as a substitute for wPLI (weighted PLI), consistent with this project's primary connectivity metric choice - not an exact replication of Bailey's method.

Alpha power (F3/F4/O1/O2), the second construct from Bailey et al., was not tested here - this check covers theta connectivity only, per the scoping decision to test the construct that drove the original ML result.

The elastic-net hyperparameter instability was diagnosed on the observed-data run only, not independently re-checked across all 1000 permutation iterations; treated as understood given the clear mechanism (grid tie-breaking under weak signal) and the well-calibrated null mean, but not exhaustively verified.